# Rebuttal Analyses

Notebook supporting reviewer responses for the EpiClass manuscript. Contains three independent analyses requested or implied by reviewer comments, all using EpiATLAS dfreeze v2 data with 100 kb genome-wide bins.

## Contents

### 1. UUID vs. EpiRR cross-validation comparison
Pairwise t-tests comparing classifier performance metrics between two splitting strategies (UUID-level and EpiRR-level grouping) for the assay and biospecimen source classifiers. Supports the methods correction disclosure regarding cross-validation grouping.

### 2. Cancer classifier breakdown by biomaterial type
Per-biomaterial-type performance evaluation of the cancer classifier, compared against a naive majority-class baseline (`DummyClassifier`). Computes accuracy and F1 macro per biomaterial type across all CV splits and visualizes them as grouped boxplots. Addresses reviewer questions about how cancer prediction performance varies across sample types.

### 3. Biospecimen source classifier breakdown by consortium
Per-project (consortium) performance evaluation of the biospecimen source classifier, computed both on the full dataset and on the Input-only subset. Each is compared to a per-project naive majority-class baseline. Addresses whether biospecimen prediction performance on input files is driven by consortia batch effect (since the input accuracy is much better than random).

## Key helper functions

- **`prepare_metrics_per_category`** — filters split predictions by a metadata column, computes per-split metrics for each category value, and returns a nested dict suitable for plotting.
- **`compute_naive_majority_metrics_per_category`** — fits a constant-prediction `DummyClassifier` per (category, split) combination to provide a baseline reference.
- **`plot_metrics_per_category`** — grouped Plotly boxplots showing model and naive baseline metrics side-by-side per category, with one subplot per metric. 

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
# pylint: disable=import-error, redefined-outer-name, use-dict-literal, too-many-lines, too-many-branches, duplicate-code, too-many-nested-blocks, missing-module-docstring
from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display
from plotly.subplots import make_subplots
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score, f1_score

from epiclass.utils.notebooks.paper.paper_utilities import (
    ASSAY,
    BIOMATERIAL_TYPE,
    CANCER,
    CELL_TYPE,
    IHECColorMap,
    MetadataHandler,
    SplitResultsHandler,
    load_split_predictions,
    pairwise_ttests,
    save_figure,
)

In [ ]:
base_dir = Path.home() / "Projects/epiclass/output/paper"
paper_dir = base_dir
if not paper_dir.exists():
    raise FileNotFoundError(f"Directory {paper_dir} does not exist.")

base_data_dir = base_dir / "data"
base_fig_dir = base_dir / "figures"

In [ ]:
IHECColorMap = IHECColorMap(base_fig_dir)
assay_colors = IHECColorMap.assay_color_map
cell_type_colors = IHECColorMap.cell_type_color_map

In [ ]:
split_results_handler = SplitResultsHandler()

metadata_handler = MetadataHandler(paper_dir)
metadata_v2 = metadata_handler.load_metadata("v2")
metadata_v2_df = metadata_v2.to_df()

In [ ]:
gen_data_dir = base_data_dir / "training_results" / "dfreeze_v2"

data_dir_100kb = gen_data_dir / "hg38_100kb_all_none"
mixed_data_dir = gen_data_dir / "mixed"

cancer_data_dir = data_dir_100kb / f"{CANCER}_1l_3000n" / "10fold-oversampling"

for path in [gen_data_dir, data_dir_100kb, mixed_data_dir, cancer_data_dir]:
    if not path.exists():
        raise FileNotFoundError(f"Directory {path} does not exist.")

In [ ]:
def change_classifier_task_name(all_metrics):
    """Remove "11c" from the classifier task names to match names for all other feature sets."""
    try:
        all_metrics["hg38_100kb_all_none"][ASSAY] = all_metrics["hg38_100kb_all_none"][  # type: ignore
            f"{ASSAY}_11c"
        ]
        del all_metrics["hg38_100kb_all_none"][f"{ASSAY}_11c"]
    except KeyError:
        pass
    return all_metrics

## Compare UUID vs EpiRR grouping metrics

In [ ]:
compare_split_method = [
    "hg38_100kb_all_none",
    "hg38_100kb_all_none_EpiRR_split",
]

In [ ]:
all_metrics = split_results_handler.obtain_all_feature_set_data(
    parent_folder=mixed_data_dir,
    merge_assays=True,
    return_type="metrics",
    include_sets=compare_split_method,
    include_categories=[ASSAY, CELL_TYPE],
    exclude_names=["16ct", "27ct", "7c", "chip-seq-only"],
    verbose=False,
)

In [ ]:
# Order the metrics
selected_metrics = {
    name: all_metrics[name]  # type: ignore
    for name in compare_split_method
    if name in all_metrics
}

selected_metrics = change_classifier_task_name(selected_metrics)

In [ ]:
df_tests = pairwise_ttests(selected_metrics)
display(df_tests)

In [ ]:
# df_tests.to_csv(Path.home() / "downloads" / "pairwise_ttests.csv", index=False)

## Cancer predictions breakdown by biomaterial type

In [ ]:
cancer_results = split_results_handler.obtain_all_feature_set_data(
    parent_folder=gen_data_dir,
    merge_assays=True,
    return_type="split_results",
    include_categories=[CANCER],
    include_sets=["hg38_100kb_all_none"],
    verbose=False,
)

cancer_split_dfs = cancer_results["hg38_100kb_all_none"][CANCER]

In [ ]:
# Load split results
cancer_concat_df_w_meta = load_split_predictions(
    data_dir=cancer_data_dir,
    metadata=metadata_v2,
    metadata_handler=metadata_handler,
    merge_assays=False,
)
print(cancer_concat_df_w_meta.shape)

In [ ]:
cancer_composition_df = (
    cancer_concat_df_w_meta.groupby([BIOMATERIAL_TYPE, "True class"])
    .size()
    .unstack(fill_value=0)
)
cancer_composition_df["total"] = cancer_composition_df.sum(axis=1)
for col in cancer_composition_df.columns[:-1]:  # skip 'total'
    cancer_composition_df[f"{col}_frac"] = (
        cancer_composition_df[col] / cancer_composition_df["total"]
    )

display(cancer_composition_df)

# cancer_composition_df.to_csv(
#     Path.home() / "downloads" / "cancer_composition.csv", float_format="%.2f"
# )

In [ ]:
def prepare_metrics_per_category(
    split_dfs: Dict,
    breakdown_col: str,
    verbose: bool = False,
) -> Dict[str, Dict[str, Dict[str, Dict[str, float]]]]:
    """Compute per-split metrics broken down by breakdown_col.

    Args:
        concat_df_w_meta: Concatenated predictions DataFrame with metadata joined.
        split_dfs: Raw split result DataFrames from read_split_results.
            Format: {split_name: DataFrame}
        breakdown_col: Column name for the category to break down by in metadata.
        verbose: Whether to print progress.

    Returns:
        Dict mapping breakdown_col -> task_category -> split_name -> metric_dict
    """
    metadata_df = metadata_handler.load_metadata("v2").to_df()
    if breakdown_col not in metadata_df.columns:
        raise ValueError(
            f"'{breakdown_col}' not found in metadata. "
            f"Available columns: {list(metadata_df.columns)}"
        )

    breakdown_order = sorted(metadata_df[breakdown_col].dropna().unique())

    md5_per_category = {
        cat: set(metadata_df.loc[metadata_df[breakdown_col] == cat].index)
        for cat in breakdown_order
    }

    if verbose:
        for btype, md5s in md5_per_category.items():
            print(f"  {btype}: {len(md5s)} samples")

    metrics_per_biomaterial = {}
    for btype in breakdown_order:
        if verbose:
            print(f"Computing metrics for biomaterial: {btype}")

        # Wrap in the classifier-level dict that compute_split_metrics expects:
        # {split_name: {classifier_name: filtered_df}}
        filtered_split_dfs = {}
        for split_name, split_df in split_dfs.items():
            filtered = split_df[split_df.index.isin(md5_per_category[btype])]
            if len(filtered) == 0:
                if verbose:
                    print(f"  WARNING: No samples for {btype} in {split_name}")
                continue
            filtered_split_dfs[split_name] = {"NN": filtered}

        if not filtered_split_dfs:
            if verbose:
                print(f"  Skipping {btype}: no samples in any split.")
            continue

        split_metrics = SplitResultsHandler().compute_split_metrics(
            all_split_dfs=filtered_split_dfs,
            split_list=list(filtered_split_dfs.keys()),
        )
        inverted = SplitResultsHandler.invert_metrics_dict(split_metrics)

        # Delete redundant AUC_macro and rename AUC_micro to AUC for clarity
        for split_name, metrics_dict in inverted["NN"].items():
            metrics_dict["AUC"] = metrics_dict.pop("AUC_micro")
            del metrics_dict["AUC_macro"]

        metrics_per_biomaterial[btype] = inverted

    return metrics_per_biomaterial

In [ ]:
cancer_metrics_per_biomaterial = prepare_metrics_per_category(
    split_dfs=cancer_split_dfs,  # type: ignore
    breakdown_col=BIOMATERIAL_TYPE,
    verbose=False,
)

In [ ]:
def plot_metrics_per_category(
    metrics_per_category: Dict[str, Dict[str, Dict[str, Dict[str, float]]]],
    category_label: str,
    naive_metrics_df: pd.DataFrame | None = None,
    naive_category_col: str | None = None,
    metrics_to_plot: List[str] | None = None,
    category_colors: Dict[str, str] | None = None,
    xaxis_label_map: Dict[str, str] | None = None,
    logdir: Path | None = None,
    name: str | None = None,
    title_prefix: str = "Classification metrics",
    filename_prefix: str = "metrics_per_category",
    y_range: Tuple[float, float] | None = None,
    boxpoints: str = "all",
    width: int = 1200,
    height: int = 800,
    scale: int = 1,
) -> None:
    """Plot per-split metrics broken down by an arbitrary category.

    Uses manual x-positioning to avoid Plotly boxmode='group' spacing bugs
    with make_subplots. Naive baseline boxes, when provided, are drawn
    adjacent to the real model boxes with consistent spacing.

    Args:
        metrics_per_category: Dict mapping category_value -> task_category
            -> split_name -> metric_dict.
        category_label: Display name for the breakdown category (used in
            titles and axis labels).
        naive_metrics_df: Optional DataFrame with naive baseline metrics.
            Must contain naive_category_col, 'split', and metric columns.
        naive_category_col: Column name in naive_metrics_df matching the
            breakdown category. Required if naive_metrics_df is provided.
        metrics_to_plot: List of metric keys to plot. Defaults to
            ["Accuracy", "F1_macro", "AUC"].
        category_colors: Optional dict mapping category values to colors.
            Defaults to Plotly qualitative palette.
        xaxis_label_map: Optional dict mapping category values to display labels for x-axis ticks.
        logdir: Directory to save figures. If None, only display.
        name: Optional suffix for titles and filenames.
        title_prefix: Prefix for the figure title.
        filename_prefix: Prefix for saved filenames.
        y_range: Optional y-axis range.
        boxpoints: "all" or "outliers".
        width: Figure width in pixels.
        height: Figure height in pixels.
        scale: Scale factor for PNG export.
    """
    if boxpoints not in ["all", "outliers"]:
        raise ValueError("Invalid boxpoints value.")
    if naive_metrics_df is not None and naive_category_col is None:
        raise ValueError(
            "naive_category_col must be provided when naive_metrics_df is given."
        )

    if metrics_to_plot is None:
        metrics_to_plot = ["Accuracy", "F1_macro", "AUC"]

    category_values = list(metrics_per_category.keys())

    if category_colors is None:
        category_colors = dict(
            zip(
                category_values,
                px.colors.qualitative.Plotly[2 : len(category_values) + 2],
            )
        )

    has_naive = naive_metrics_df is not None
    n_slots = 2 if has_naive else 1
    group_width = n_slots + 0.8  # spacing between category groups
    box_width = 0.6

    x_positions = {cat_val: i * group_width for i, cat_val in enumerate(category_values)}

    reference_cat = next(iter(metrics_per_category))
    classifier_names = list(metrics_per_category[reference_cat].keys())

    for algo in classifier_names:
        fig = make_subplots(
            rows=1,
            cols=len(metrics_to_plot),
            shared_yaxes=True,
            subplot_titles=[m.replace("_", " ") for m in metrics_to_plot],
            x_title=category_label,
            y_title="Metric Value",
            horizontal_spacing=0.05,
        )

        for col_idx, metric in enumerate(metrics_to_plot, start=1):
            for cat_val in category_values:
                color = category_colors.get(cat_val, "grey")
                x_pos = x_positions[cat_val]

                # --- Model box ---
                y_vals = []
                hovertext = []
                if algo in metrics_per_category[cat_val]:
                    split_dict = metrics_per_category[cat_val][algo]
                    for split_name, metric_dict in split_dict.items():
                        if metric not in metric_dict:
                            continue
                        val = metric_dict[metric]
                        if val is None or (isinstance(val, float) and np.isnan(val)):
                            continue
                        y_vals.append(val)
                        hovertext.append(f"{cat_val} - {split_name}: {val:.4f}")

                if y_vals:
                    fig.add_trace(
                        go.Box(
                            x=[x_pos] * len(y_vals),
                            y=y_vals,
                            name=cat_val,
                            boxmean=True,
                            boxpoints=boxpoints,
                            width=box_width,
                            marker=dict(
                                size=4, color=color, line=dict(color="black", width=0.5)
                            ),
                            line=dict(width=1, color="black"),
                            fillcolor=color,
                            opacity=0.7,
                            hovertemplate="%{text}",
                            text=hovertext,
                            legendgroup=cat_val,
                            showlegend=False,
                        ),
                        row=1,
                        col=col_idx,
                    )

                # --- Naive baseline box ---
                if has_naive:
                    cat_naive = naive_metrics_df[
                        naive_metrics_df[naive_category_col] == cat_val
                    ]
                    naive_vals = []
                    naive_hover = []
                    for _, row in cat_naive.iterrows():
                        val = row.get(metric)
                        if val is None or pd.isna(val):
                            continue
                        naive_vals.append(val)
                        naive_hover.append(
                            f"Naive - {cat_val} - {row['split']}: {val:.4f}"
                        )

                    if naive_vals:
                        fig.add_trace(
                            go.Box(
                                x=[x_pos + 1] * len(naive_vals),
                                y=naive_vals,
                                name="Naive baseline",
                                boxmean=True,
                                boxpoints=boxpoints,
                                width=box_width,
                                marker=dict(
                                    size=4,
                                    color="red",
                                    line=dict(color="black", width=0.5),
                                ),
                                line=dict(width=1, color="black"),
                                fillcolor="rgba(255, 0, 0, 0.15)",
                                hovertemplate="%{text}",
                                text=naive_hover,
                                legendgroup="naive_baseline",
                                showlegend=False,
                            ),
                            row=1,
                            col=col_idx,
                        )

            # X-axis ticks centered on each group
            tick_offset = 0.5 if has_naive else 0

            tick_labels = [
                xaxis_label_map.get(cat, cat) if xaxis_label_map else cat
                for cat in category_values
            ]
            tick_labels = [label.replace(" ", "<br>") for label in tick_labels]

            fig.update_xaxes(
                tickmode="array",
                tickvals=[x_positions[cat] + tick_offset for cat in category_values],
                ticktext=tick_labels,
                row=1,
                col=col_idx,
            )

        # --- Legend entries ---
        for cat_val in category_values:
            color = category_colors.get(cat_val, "grey")
            fig.add_trace(
                go.Scatter(
                    x=[None],
                    y=[None],
                    mode="markers",
                    name=cat_val,
                    marker=dict(size=8, color=color),
                    legendgroup=cat_val,
                    showlegend=True,
                )
            )

        if has_naive:
            fig.add_trace(
                go.Box(
                    y=[None],
                    name="Naive baseline",
                    marker=dict(color="red"),
                    line=dict(color="red"),
                    fillcolor="rgba(255, 0, 0, 0.15)",
                    legendgroup="naive_baseline",
                    showlegend=True,
                )
            )

        # --- Layout ---
        title = f"{title_prefix} per {category_label.lower()}"
        if name is not None:
            title += f" - {name}"

        fig.update_layout(
            width=width,
            height=height,
            title=title,
            legend=dict(itemsizing="constant"),
        )

        if y_range is not None:
            fig.update_yaxes(range=y_range)

        if logdir:
            logdir = Path(logdir)
            logdir.mkdir(parents=True, exist_ok=True)
            base_name = f"{filename_prefix}_{algo}"
            if name is not None:
                base_name += f"_{name}"
            save_figure(fig, logdir, base_name, scale=scale)

        fig.show()

In [ ]:
def compute_naive_majority_metrics_per_category(
    split_dfs: Dict[str, pd.DataFrame],
    breakdown_col: str,
    breakdown_order: List[str] | None = None,
    verbose: bool = False,
) -> pd.DataFrame:
    """Compute metrics for a naive majority-class classifier, per category and split.

    Args:
        split_dfs: Raw split result DataFrames from read_split_results.
            Format: {split_name: DataFrame}
        breakdown_col: Metadata column name to break down by.
        breakdown_order: Optional ordered list of category values to include.
            If None, all unique values are used in sorted order.
        verbose: Whether to print progress.

    Returns:
        DataFrame with columns: breakdown_col, split, n_samples, majority_class,
            majority_fraction, Accuracy, F1_macro, AUC_micro, AUC_macro
    """
    metadata_df = metadata_handler.load_metadata("v2").to_df()
    if breakdown_col not in metadata_df.columns:
        raise ValueError(
            f"'{breakdown_col}' not found in metadata. "
            f"Available columns: {list(metadata_df.columns)}"
        )

    if breakdown_order is None:
        breakdown_order = sorted(metadata_df[breakdown_col].dropna().unique())

    md5_per_category = {
        cat: set(metadata_df.loc[metadata_df[breakdown_col] == cat].index)
        for cat in breakdown_order
    }

    rows = []
    for cat in breakdown_order:
        for split_name, split_df in split_dfs.items():
            filtered = split_df[split_df.index.isin(md5_per_category[cat])]
            if len(filtered) == 0:
                continue

            y_true = filtered["True class"]
            n_samples = len(y_true)
            X_dummy = np.zeros((n_samples, 1))

            majority_class_value = y_true.value_counts().idxmax()
            dummy = DummyClassifier(strategy="constant", constant=majority_class_value)
            dummy.fit(X_dummy, y_true)
            y_pred = dummy.predict(X_dummy)

            majority_fraction = (y_true == majority_class_value).mean()
            present_classes = sorted(y_true.unique())

            accuracy = accuracy_score(y_true, y_pred)
            f1_macro = f1_score(
                y_true, y_pred, labels=present_classes, average="macro", zero_division=0
            )

            rows.append(
                {
                    breakdown_col: cat,
                    "split": split_name,
                    "n_samples": n_samples,
                    "majority_class": majority_class_value,
                    "majority_fraction": majority_fraction,
                    "Accuracy": accuracy,
                    "F1_macro": f1_macro,
                }
            )

            if verbose:
                print(
                    f"  {cat} | {split_name}: n={n_samples}, "
                    f"majority={majority_class_value} ({majority_fraction:.1%}), "
                    f"acc={accuracy:.4f}, f1={f1_macro:.4f}"
                )

    return pd.DataFrame(rows)

In [ ]:
naive_metrics_df = compute_naive_majority_metrics_per_category(
    split_dfs=cancer_split_dfs,  # type: ignore
    breakdown_col=BIOMATERIAL_TYPE,
    verbose=False,
)

In [ ]:
plot_metrics_per_category(
    metrics_per_category=cancer_metrics_per_biomaterial,
    category_label="Biomaterial Type",
    naive_metrics_df=naive_metrics_df,
    naive_category_col=BIOMATERIAL_TYPE,
    y_range=(0.34, 1.01),
    metrics_to_plot=["Accuracy", "F1_macro"],
    logdir=Path.home() / "downloads",
    filename_prefix="cancer_metrics_per_biomaterial",
)

## Biospecimen predictions per consortia

In [ ]:
data_dir = data_dir_100kb / f"{CELL_TYPE}_1l_3000n" / "10fold-oversampling"
if not data_dir.exists():
    raise FileNotFoundError(f"Directory {data_dir} does not exist.")

In [ ]:
cell_type_results = split_results_handler.obtain_all_feature_set_data(
    parent_folder=gen_data_dir,
    merge_assays=True,
    return_type="split_results",
    include_categories=[CELL_TYPE],
    include_sets=["hg38_100kb_all_none"],
    verbose=False,
)

cell_type_split_dfs = cell_type_results["hg38_100kb_all_none"][CELL_TYPE]

In [ ]:
input_mask = metadata_v2_df[ASSAY] == "input"
input_md5s = set(metadata_v2_df.loc[input_mask].index)

In [ ]:
cell_type_results_input = {}
for split_name, split_df in cell_type_split_dfs.items():
    print(f"Processing split: {split_name}")
    print(f"  Total samples in split: {len(split_df)}")
    input_df = split_df[split_df.index.isin(input_md5s)]
    print(f"  Samples with 'input' assay in split: {len(input_df)}")
    if len(input_df) == 0:
        print(f"  WARNING: No input samples in {split_name}")
        continue
    cell_type_results_input[split_name] = input_df

In [ ]:
cell_type_metrics_breakdown = prepare_metrics_per_category(
    split_dfs=cell_type_split_dfs,  # type: ignore
    breakdown_col="project",
    verbose=True,
)

In [ ]:
cell_type_metrics_breakdown_input = prepare_metrics_per_category(
    split_dfs=cell_type_results_input,  # type: ignore
    breakdown_col="project",
    verbose=False,
)

In [ ]:
naive_metrics_df = compute_naive_majority_metrics_per_category(
    split_dfs=cell_type_split_dfs,  # type: ignore
    breakdown_col="project",
    verbose=False,
)

In [ ]:
naive_metrics_df_input = compute_naive_majority_metrics_per_category(
    split_dfs=cell_type_results_input,  # type: ignore
    breakdown_col="project",
    verbose=False,
)

In [ ]:
label_remapper = {
    "NIH Roadmap Epigenomics": "Roadmap",
    "Korea Epigenome Project (KNIH)": "KNIH",
}

In [ ]:
# all assays
plot_metrics_per_category(
    metrics_per_category=cell_type_metrics_breakdown,
    category_label="Project",
    naive_metrics_df=naive_metrics_df,
    naive_category_col="project",
    metrics_to_plot=["Accuracy", "F1_macro"],
    xaxis_label_map=label_remapper,
    # logdir=Path.home() / "downloads",
    # filename_prefix="cell_type_metrics_per_project",
)

In [ ]:
# input only
plot_metrics_per_category(
    metrics_per_category=cell_type_metrics_breakdown_input,
    category_label="Project",
    naive_metrics_df=naive_metrics_df_input,
    naive_category_col="project",
    metrics_to_plot=["Accuracy", "F1_macro"],
    xaxis_label_map=label_remapper,
    # logdir=Path.home() / "downloads",
    # filename_prefix="cell_type_metrics_per_project_input_only",
)